In [3]:
# Q1) How many new Free and Paid accounts were created in the week of 3/3–3/7?
# - Uses startdate to define "new account created"
# - Uses the user's plan on their first observed activity day as the "plan at creation"
# - Produces an interactive Plotly bar chart with labels

import pandas as pd
import numpy as np
import plotly.graph_objects as go

In [33]:
# Plotly Sankey with a dropdown to switch between:
# 1) "Calendar" view: within calendar window 3/3–3/7
# 2) "Lifecycle" view: within each user's first week (day 0–6 from startdate)
#
# Sankey structure (both views):
#   New Accounts (created 3/3–3/7) -> Start Plan (Free/Paid/Unknown) -> End-of-week Plan (Free/Paid/Unknown)

import pandas as pd
import numpy as np
import plotly.graph_objects as go

DATA_PATH = "/Users/cloudmaple/Downloads/Microsoft_Interview/Tech_Coding/interviewCase_IDE_expanded_multiweek.csv"

COHORT_START = pd.Timestamp("2025-03-03")
COHORT_END   = pd.Timestamp("2025-03-07")

# -------------------------
# 1) Load + clean
# -------------------------
df = pd.read_csv(DATA_PATH)

df["userid"] = df["userid"].astype(str).str.strip()

df["date"] = pd.to_datetime(df["date"], errors="coerce", format="mixed")
df["startdate"] = pd.to_datetime(df["startdate"], errors="coerce", format="mixed")

df["plan"] = df["plan"].astype(str).str.strip().str.title()
df.loc[~df["plan"].isin(["Free", "Paid"]), "plan"] = np.nan

# Numeric (optional)
df["DaysSinceStart"] = pd.to_numeric(df.get("DaysSinceStart"), errors="coerce")

# Keep only rows with essential fields
df = df.dropna(subset=["userid", "date", "startdate"]).copy()

# Derive day index (lifecycle)
mask = df["date"].notna() & df["startdate"].notna()
df.loc[mask, "day"] = (df.loc[mask, "date"] - df.loc[mask, "startdate"]).dt.days
df["day"] = df["day"].where(df["day"].notna(), df["DaysSinceStart"])

# Sanity filter day (optional but recommended for messy data)
df.loc[~df["day"].between(-5, 365), "day"] = np.nan

# -------------------------
# 2) Identify new accounts created in 3/3–3/7 (cohort users)
# -------------------------
user_start = (
    df.groupby("userid", as_index=False)
      .agg(start_effective=("startdate", "min"))
)

new_user_ids = set(
    user_start.loc[user_start["start_effective"].between(COHORT_START, COHORT_END), "userid"]
)

cohort = df[df["userid"].isin(new_user_ids)].copy()

# -------------------------
# 3) Helper: compute (start_plan, end_week_plan) per user for a given view
# -------------------------
def compute_user_state(view: str) -> pd.DataFrame:
    """
    view:
      - "calendar": only events with date in [COHORT_START, COHORT_END]
      - "lifecycle": only events with day in [0, 6] from each user's startdate
    returns: DataFrame with columns [userid, start_plan, end_week_plan]
    """
    if view == "calendar":
        wk = cohort[cohort["date"].between(COHORT_START, COHORT_END)].copy()
        sort_cols = ["userid", "date"]
    elif view == "lifecycle":
        wk = cohort[cohort["day"].between(0, 6)].copy()
        sort_cols = ["userid", "day", "date"]
    else:
        raise ValueError("view must be 'calendar' or 'lifecycle'")

    # If a user has no rows in the selected window, they would disappear.
    # In practice, for this dataset they should have at least day 0,
    # but we'll handle it by allowing Unknown.
    wk = wk.sort_values(sort_cols)

    # Forward-fill plan within user inside the selected window
    wk["plan_ffill"] = wk.groupby("userid")["plan"].ffill()

    # Start plan: earliest non-null plan within the view window
    start_plan = (
        wk.groupby("userid")["plan_ffill"]
          .apply(lambda s: s.dropna().iloc[0] if len(s.dropna()) > 0 else "Unknown")
    )

    # End-of-week plan definition differs slightly by view:
    if view == "calendar":
        # "End of week" = last observed plan on/before 3/7 within the calendar window
        end_plan = (
            wk.groupby("userid")["plan_ffill"]
              .apply(lambda s: s.dropna().iloc[-1] if len(s.dropna()) > 0 else "Unknown")
        )
    else:
        # "End of lifecycle week" = prefer day 6 if present, else last observed <=6
        # We'll implement as: take last plan_ffill after sorting by (day, date).
        end_plan = (
            wk.groupby("userid")["plan_ffill"]
              .apply(lambda s: s.dropna().iloc[-1] if len(s.dropna()) > 0 else "Unknown")
        )

    user_state = pd.DataFrame({
        "userid": start_plan.index,
        "start_plan": start_plan.values,
        "end_week_plan": end_plan.values
    })

    # Ensure only Free/Paid/Unknown
    for c in ["start_plan", "end_week_plan"]:
        user_state[c] = user_state[c].where(user_state[c].isin(["Free", "Paid"]), "Unknown")

    # Include any cohort users missing from wk (no rows in window) as Unknown->Unknown
    missing = list(new_user_ids - set(user_state["userid"]))
    if missing:
        user_state = pd.concat(
            [user_state,
             pd.DataFrame({"userid": missing, "start_plan": "Unknown", "end_week_plan": "Unknown"})],
            ignore_index=True
        )

    return user_state

# -------------------------
# 4) Helper: build Sankey link arrays from user_state
# -------------------------
nodes = [
    "New Accounts (3/3–3/7)",
    "Start: Free", "Start: Paid", "Start: Unknown",
    "End Week: Free", "End Week: Paid", "End Week: Unknown"
]
node_idx = {n: i for i, n in enumerate(nodes)}

def build_links(user_state: pd.DataFrame):
    links = []

    def add(src, tgt, val):
        if val and val > 0:
            links.append((node_idx[src], node_idx[tgt], int(val)))

    # New -> Start
    start_counts = user_state["start_plan"].value_counts(dropna=False)
    add("New Accounts (3/3–3/7)", "Start: Free", start_counts.get("Free", 0))
    add("New Accounts (3/3–3/7)", "Start: Paid", start_counts.get("Paid", 0))
    add("New Accounts (3/3–3/7)", "Start: Unknown", start_counts.get("Unknown", 0))

    # Start -> End
    trans = user_state.groupby(["start_plan", "end_week_plan"]).size().reset_index(name="users")
    for _, r in trans.iterrows():
        add(f"Start: {r['start_plan']}", f"End Week: {r['end_week_plan']}", r["users"])

    source = [s for s, t, v in links]
    target = [t for s, t, v in links]
    value  = [v for s, t, v in links]
    return source, target, value

# -------------------------
# 5) Build both views
# -------------------------
state_calendar  = compute_user_state("calendar")
state_lifecycle = compute_user_state("lifecycle")

src_cal, tgt_cal, val_cal = build_links(state_calendar)
src_lc,  tgt_lc,  val_lc  = build_links(state_lifecycle)

# -------------------------
# 6) Create Sankey with dropdown
# -------------------------
fig = go.Figure(data=[go.Sankey(
    node=dict(label=nodes, pad=18, thickness=18),
    link=dict(source=src_lc, target=tgt_lc, value=val_lc)  # default: lifecycle
)])

fig.update_layout(
    title="Sankey (Lifecycle view): New accounts 3/3–3/7 → Start plan → End of first 7 days (Day 0–6)",
    font_size=12,
    margin=dict(t=80, l=30, r=30, b=30),
    updatemenus=[
        dict(
            type="dropdown",
            x=0.0,
            y=1.1,
            buttons=[
                dict(
                    label="Lifecycle view (day 0–6)",
                    method="update",
                    args=[
                        {"link": [{"source": src_lc, "target": tgt_lc, "value": val_lc}]},
                        {"title": "Sankey (Lifecycle view): New accounts 3/3–3/7 → Start plan → End of first 7 days (Day 0–6)"}
                    ],
                ),
                dict(
                    label="Calendar view (3/3–3/7)",
                    method="update",
                    args=[
                        {"link": [{"source": src_cal, "target": tgt_cal, "value": val_cal}]},
                        {"title": "Sankey (Calendar view): New accounts 3/3–3/7 → Start plan → Plan by end of 3/7"}
                    ],
                ),
            ],
        )
    ],
)

fig.show()


In [34]:
df.head()

,date,userid,startdate,plan,Status,DayOfWeek,Weekend,DaysSinceStart,AI_AutoComplete_Success,AI_AutoComplete_Error,AI_Chat_Success,AI_Chat_Error,AI_Agent_Success,AI_Agent_Error,day
0,2025-05-08,U0054,2025-03-15,Paid,Cancelled,Thursday,False,54,4,1,3,0,2,0,54.0
1,2025-05-06,U0158,2025-03-28,Free,Active,Tuesday,False,39,9,0,2,0,0,0,39.0
2,2025-06-17,U0245,2025-04-20,Paid,Cancelled,Tuesday,False,58,1,1,2,2,2,0,58.0
3,2025-04-11,U0073,2025-03-15,Paid,Active,Friday,False,27,7,1,2,0,0,0,27.0
4,2025-03-17,U0026,2025-03-05,Paid,Active,Monday,False,12,5,0,3,1,1,0,12.0


In [35]:
df['Status'].unique()

array(['Cancelled', 'Active'], dtype=object)

In [36]:
import pandas as pd
import numpy as np
import plotly.graph_objects as go


# -----------------------------
# 1) Load + Clean
# -----------------------------
df = pd.read_csv(DATA_PATH)

df["userid"] = df["userid"].astype(str).str.strip()
df["date"] = pd.to_datetime(df["date"], errors="coerce", format="mixed")
df["startdate"] = pd.to_datetime(df["startdate"], errors="coerce", format="mixed")

df["plan"] = df["plan"].astype(str).str.strip().str.title()
df.loc[~df["plan"].isin(["Free", "Paid"]), "plan"] = np.nan

df["Status"] = df["Status"].astype(str).str.strip().strtitle() if hasattr(str, "strtitle") else df["Status"].astype(str).str.strip().str.title()
df.loc[~df["Status"].isin(["Active", "Cancelled"]), "Status"] = np.nan

df = df.dropna(subset=["userid", "date", "startdate"]).copy()

# -----------------------------
# 2) Define 8 calendar weeks starting W0 = week of 2025/03/03 (Monday-based)
# -----------------------------
BASE_WEEK_START = pd.Timestamp("2025-03-03")  # W0 start (Monday)
N_WEEKS = 8

def week_start_monday(ts: pd.Timestamp) -> pd.Timestamp:
    ts = ts.normalize()
    return ts - pd.Timedelta(days=ts.weekday())

# Cohort week index per user (based on startdate week)
# NEW signups in week w = users whose startdate is in [Ww_start, Ww_start+6]
user_start = (
    df.groupby("userid", as_index=False)
      .agg(start_effective=("startdate", "min"))  # should be stable per user
)

user_start["signup_week_start"] = user_start["start_effective"].apply(week_start_monday)
user_start["cohort_week"] = ((user_start["signup_week_start"] - BASE_WEEK_START) / pd.Timedelta(days=7)).astype(int)

# Keep only cohorts in W0..W7
user_start = user_start[user_start["cohort_week"].between(0, N_WEEKS-1)].copy()

cohort_users = set(user_start["userid"])
df = df[df["userid"].isin(cohort_users)].copy()

# -----------------------------
# 3) Segment users by START PLAN (Free/Paid at earliest observed activity)
# -----------------------------
user_first_event = (
    df.sort_values(["userid", "date"])
      .groupby("userid", as_index=False)
      .first()[["userid", "plan"]]
      .rename(columns={"plan": "start_plan"})
)

user_dim = user_start.merge(user_first_event, on="userid", how="left")
user_dim["start_plan"] = user_dim["start_plan"].where(user_dim["start_plan"].isin(["Free", "Paid"]), "Unknown")

# -----------------------------
# 4) Build end-of-week Status per user per calendar week
#    - If multiple status in a week, take the LAST status (end-of-week) by date
#    - Users may churn then rejoin; we keep each week’s final status
# -----------------------------
# Assign each event row to its calendar week end (Sunday)
df = df.sort_values(["userid", "date"]).copy()
df["week_end"] = (df["date"] - pd.to_timedelta(df["date"].dt.weekday, unit="D") + pd.Timedelta(days=6)).dt.normalize()
# The above makes week_end = Sunday of that week.

# Limit to our 8-week horizon (week_end range)
week_ends = [BASE_WEEK_START + pd.Timedelta(days=6 + 7*i) for i in range(N_WEEKS)]
week_end_set = set(week_ends)
df = df[df["week_end"].isin(week_end_set)].copy()

week_end_to_idx = {we: i for i, we in enumerate(week_ends)}
df["week_idx"] = df["week_end"].map(week_end_to_idx)

# End-of-week status: last status in that week (per user, per week_idx)
weekly_status = (
    df.dropna(subset=["Status"])
      .groupby(["userid", "week_idx"], as_index=False)
      .agg(end_week_status=("Status", "last"))  # 'last' after sorting by date
)

# Expand to full grid (userid x week_idx) so missing weeks appear (optional)
# Then forward-fill across weeks so billing state persists if we lack a record that week.
all_users = user_dim["userid"].unique()
grid = pd.MultiIndex.from_product([all_users, range(N_WEEKS)], names=["userid", "week_idx"]).to_frame(index=False)
weekly_status = grid.merge(weekly_status, on=["userid", "week_idx"], how="left")

weekly_status = weekly_status.sort_values(["userid", "week_idx"])
weekly_status["end_week_status"] = weekly_status.groupby("userid")["end_week_status"].ffill()

# Join cohort + start_plan
weekly_status = weekly_status.merge(
    user_dim[["userid", "cohort_week", "start_plan"]],
    on="userid",
    how="left"
)

weekly_status["is_active_eow"] = (weekly_status["end_week_status"] == "Active")
weekly_status["is_cancelled_eow"] = (weekly_status["end_week_status"] == "Cancelled")

# -----------------------------
# 5) Build upper-triangle cohort table (counts + rates)
#    - Diagonal (Wk,Wk) = NEW signups
#    - Right cells = active_count (retention_rate) at end of that week
#    - Left cells blank
# -----------------------------
def build_cohort_table(segment: str):
    """
    segment: "All" | "Free" | "Paid"
    Returns: plotly Figure (go.Table)
    """
    d = weekly_status.copy()
    u = user_dim.copy()

    if segment in ["Free", "Paid"]:
        u = u[u["start_plan"] == segment].copy()
        d = d[d["start_plan"] == segment].copy()

    # cohort sizes (new signups each week)
    cohort_sizes = (
        u.groupby("cohort_week")["userid"]
         .nunique()
         .reindex(range(N_WEEKS), fill_value=0)
    )

    # Active counts by (cohort_week, week_idx)
    active_counts = (
        d.groupby(["cohort_week", "week_idx"])["is_active_eow"]
         .sum()
         .unstack("week_idx")
         .reindex(index=range(N_WEEKS), columns=range(N_WEEKS), fill_value=0)
    )

    # Cancelled counts by (cohort_week, week_idx) (optional insight)
    cancelled_counts = (
        d.groupby(["cohort_week", "week_idx"])["is_cancelled_eow"]
         .sum()
         .unstack("week_idx")
         .reindex(index=range(N_WEEKS), columns=range(N_WEEKS), fill_value=0)
    )

    # Build display matrix (strings)
    matrix = []
    for cw in range(N_WEEKS):
        row = []
        n_new = int(cohort_sizes.loc[cw])

        for w in range(N_WEEKS):
            if w < cw:
                row.append("")  # upper-triangle requirement
                continue

            if w == cw:
                # Diagonal cell: NEW signups count
                row.append(f"{n_new}")
                continue

            # Retention: active at end of week w among cohort cw
            if n_new == 0:
                row.append("")
            else:
                a = int(active_counts.loc[cw, w])
                r = a / n_new
                # optional: include cancelled count too
                c = int(cancelled_counts.loc[cw, w])
                row.append(f"{a} ({r*100:.1f}%) | C:{c}")
        matrix.append(row)

    # Row labels: cohort week + start date
    row_labels = []
    for cw in range(N_WEEKS):
        wk_start = (BASE_WEEK_START + pd.Timedelta(days=7*cw)).date()
        row_labels.append(f"W{cw} ({wk_start})")

    col_labels = []
    for w in range(N_WEEKS):
        wk_start = (BASE_WEEK_START + pd.Timedelta(days=7*w)).date()
        col_labels.append(f"W{w} ({wk_start})")

    # Build Plotly Table
    header = ["Cohort"] + col_labels
    cells = [row_labels] + list(map(list, zip(*matrix)))  # transpose matrix columns into table columns

    fig = go.Figure(data=[go.Table(
        header=dict(values=header, align="center"),
        cells=dict(values=cells, align="center")
    )])

    fig.update_layout(
        title=f"Cohort Retention Table (End-of-Week Status) — {segment} (8 weeks)  |  Cell = Active (Rate) | C:Cancelled",
        margin=dict(t=70, l=20, r=20, b=20)
    )
    return fig

# -----------------------------
# 6) Produce 3 views
# -----------------------------
fig_all = build_cohort_table("All")
fig_free = build_cohort_table("Free")
fig_paid = build_cohort_table("Paid")

fig_all.show()
fig_free.show()
fig_paid.show()

# Optional sanity check: new signups by cohort week
print("New signups by cohort week (All):")
print(user_dim.groupby("cohort_week")["userid"].nunique().reindex(range(N_WEEKS), fill_value=0))


New signups by cohort week (All):
cohort_week
0    40
1    40
2    40
3    40
4    40
5    40
6    40
7    40
Name: userid, dtype: int64


In [38]:
# Q3) Free → Paid conversion (W0 cohort) + Sankey with dropdown for end-week (e.g., W5)
# Cohort: users who SIGNED UP in W0 (week starting 2025-03-03) AND started as Free.
# For a selected end week Wx, we classify each user at END OF WEEK (last status/plan in that week):
#   - If end_week_status == "Cancelled"  -> "Cancelled"
#   - Else if end_week_plan == "Paid"    -> "Paid"
#   - Else                                "Free"   (default)
#
# Sankey shows: "W0 Free signups" -> (Free, Paid, Cancelled) at the selected end week.
# Dropdown lets you pick end week W0..W7.

import pandas as pd
import numpy as np
import plotly.graph_objects as go


BASE_WEEK_START = pd.Timestamp("2025-03-03")  # W0 start (Monday)
N_WEEKS = 8

# -----------------------------
# 1) Load + clean
# -----------------------------
df = pd.read_csv(DATA_PATH)

df["userid"] = df["userid"].astype(str).str.strip()
df["date"] = pd.to_datetime(df["date"], errors="coerce", format="mixed")
df["startdate"] = pd.to_datetime(df["startdate"], errors="coerce", format="mixed")

df["plan"] = df["plan"].astype(str).str.strip().str.title()
df.loc[~df["plan"].isin(["Free", "Paid"]), "plan"] = np.nan

df["Status"] = df["Status"].astype(str).str.strip().str.title()
df.loc[~df["Status"].isin(["Active", "Cancelled"]), "Status"] = np.nan

df = df.dropna(subset=["userid", "date", "startdate"]).copy()
df = df.sort_values(["userid", "date"]).copy()

def week_start_monday(ts: pd.Timestamp) -> pd.Timestamp:
    ts = ts.normalize()
    return ts - pd.Timedelta(days=ts.weekday())

# -----------------------------
# 2) User-level dimension: cohort week + start plan
# -----------------------------
user_start = (
    df.groupby("userid", as_index=False)
      .agg(start_effective=("startdate", "min"))
)

user_start["signup_week_start"] = user_start["start_effective"].apply(week_start_monday)
user_start["cohort_week"] = ((user_start["signup_week_start"] - BASE_WEEK_START) / pd.Timedelta(days=7)).astype(int)

# Keep only users in W0..W7 horizon (optional)
user_start = user_start[user_start["cohort_week"].between(0, N_WEEKS-1)].copy()
df = df[df["userid"].isin(set(user_start["userid"]))].copy()

user_first_event = (
    df.sort_values(["userid", "date"])
      .groupby("userid", as_index=False)
      .first()[["userid", "plan"]]
      .rename(columns={"plan": "start_plan"})
)
user_first_event["start_plan"] = user_first_event["start_plan"].where(
    user_first_event["start_plan"].isin(["Free", "Paid"]), "Unknown"
)

user_dim = user_start.merge(user_first_event, on="userid", how="left")

# -----------------------------
# 3) Define the W0 Free-starter cohort
# -----------------------------
w0_free_users = set(
    user_dim.loc[(user_dim["cohort_week"] == 0) & (user_dim["start_plan"] == "Free"), "userid"]
)
print("W0 Free starters:", len(w0_free_users))

# -----------------------------
# 4) Compute end-of-week plan & status per user per week (0..7)
#     - Use LAST record in the week (after sorting by date)
#     - Forward-fill across weeks so state persists if a week has no records
# -----------------------------
# Map each row to its week_end (Sunday) and week_idx
week_ends = [BASE_WEEK_START + pd.Timedelta(days=6 + 7*i) for i in range(N_WEEKS)]
week_end_set = set(week_ends)
week_end_to_idx = {we: i for i, we in enumerate(week_ends)}

# Sunday week_end for each date
df["week_end"] = (df["date"] - pd.to_timedelta(df["date"].dt.weekday, unit="D") + pd.Timedelta(days=6)).dt.normalize()
df = df[df["week_end"].isin(week_end_set)].copy()
df["week_idx"] = df["week_end"].map(week_end_to_idx)

# Restrict to cohort users (W0 Free)
d = df[df["userid"].isin(w0_free_users)].copy()
d = d.sort_values(["userid", "week_idx", "date"])

# Last plan/status within each week (end-of-week snapshot from event logs)
snap = (
    d.groupby(["userid", "week_idx"], as_index=False)
     .agg(end_week_plan=("plan", "last"),
          end_week_status=("Status", "last"))
)

# Expand full grid to include missing weeks per user
all_users = sorted(w0_free_users)
grid = pd.MultiIndex.from_product([all_users, range(N_WEEKS)], names=["userid", "week_idx"]).to_frame(index=False)
snap = grid.merge(snap, on=["userid", "week_idx"], how="left").sort_values(["userid", "week_idx"])

# Forward-fill across weeks (state persists)
snap["end_week_plan"] = snap.groupby("userid")["end_week_plan"].ffill()
snap["end_week_status"] = snap.groupby("userid")["end_week_status"].ffill()

# Normalize unknowns
snap["end_week_plan"] = snap["end_week_plan"].where(snap["end_week_plan"].isin(["Free", "Paid"]), "Free")
snap["end_week_status"] = snap["end_week_status"].where(snap["end_week_status"].isin(["Active", "Cancelled"]), "Active")

# -----------------------------
# 5) Classify final state for a given end week
# -----------------------------
def classify_state(plan: str, status: str) -> str:
    if status == "Cancelled":
        return "Cancelled"
    if plan == "Paid":
        return "Paid"
    return "Free"

def counts_for_week(week_idx: int):
    s = snap[snap["week_idx"] == week_idx].copy()
    s["state"] = [classify_state(p, st) for p, st in zip(s["end_week_plan"], s["end_week_status"])]
    counts = s["state"].value_counts().to_dict()
    free_n = int(counts.get("Free", 0))
    paid_n = int(counts.get("Paid", 0))
    canc_n = int(counts.get("Cancelled", 0))
    return free_n, paid_n, canc_n

# Conversion count/rate by selected week: Paid OR (optionally) ever paid up to that week.
# Here: "converted by week Wx" = end-of-week state is Paid at Wx.
def conversion_by_week(week_idx: int):
    _, paid_n, _ = counts_for_week(week_idx)
    cohort_n = len(w0_free_users)
    rate = paid_n / cohort_n if cohort_n else np.nan
    return paid_n, rate

# -----------------------------
# 6) Build Sankey with dropdown
# -----------------------------
nodes = [
    "W0 Free signups",
    "Free", "Paid", "Cancelled"
]
node_idx = {n: i for i, n in enumerate(nodes)}

def sankey_links_for_week(week_idx: int):
    free_n, paid_n, canc_n = counts_for_week(week_idx)
    source = [node_idx["W0 Free signups"]] * 3
    target = [node_idx["Free"], node_idx["Paid"], node_idx["Cancelled"]]
    value  = [free_n, paid_n, canc_n]
    return source, target, value

# Default view: W5 if available else last week
default_week = 5 if N_WEEKS > 5 else (N_WEEKS - 1)

src0, tgt0, val0 = sankey_links_for_week(default_week)
paid_conv, conv_rate = conversion_by_week(default_week)
print(f"By end of W{default_week}: Converted to Paid = {paid_conv} / {len(w0_free_users)} ({conv_rate*100:.1f}%)")

fig = go.Figure(data=[go.Sankey(
    node=dict(label=nodes, pad=18, thickness=18),
    link=dict(source=src0, target=tgt0, value=val0)
)])

buttons = []
for w in range(N_WEEKS):
    src, tgt, val = sankey_links_for_week(w)
    paid_conv, conv_rate = conversion_by_week(w)
    title = (
        f"W0 Free cohort → State at end of W{w} | "
        f"Paid (converted) = {paid_conv}/{len(w0_free_users)} ({(conv_rate*100 if len(w0_free_users) else 0):.1f}%)"
    )
    buttons.append(dict(
        label=f"End week W{w}",
        method="update",
        args=[
            {"link": [{"source": src, "target": tgt, "value": val}]},
            {"title": title}
        ],
    ))

fig.update_layout(
    title=f"W0 Free cohort → State at end of W{default_week}",
    font_size=12,
    margin=dict(t=80, l=30, r=30, b=30),
    updatemenus=[dict(
        type="dropdown",
        x=0.0,
        y=1.1,
        buttons=buttons
    )]
)

fig.show()


W0 Free starters: 32
By end of W5: Converted to Paid = 22 / 32 (68.8%)


In [39]:
import pandas as pd
import numpy as np
import plotly.graph_objects as go


BASE_WEEK_START = pd.Timestamp("2025-03-03")  # W0 start
N_WEEKS = 8

# -----------------------------
# 1) Load + clean
# -----------------------------
df = pd.read_csv(DATA_PATH)

df["userid"] = df["userid"].astype(str).str.strip()
df["date"] = pd.to_datetime(df["date"], errors="coerce", format="mixed")
df["startdate"] = pd.to_datetime(df["startdate"], errors="coerce", format="mixed")

df["plan"] = df["plan"].astype(str).str.strip().str.title()
df.loc[~df["plan"].isin(["Free", "Paid"]), "plan"] = np.nan

df["Status"] = df["Status"].astype(str).str.strip().str.title()
df.loc[~df["Status"].isin(["Active", "Cancelled"]), "Status"] = np.nan

df = df.dropna(subset=["userid", "date", "startdate"]).copy()
df = df.sort_values(["userid", "date"])

def week_start_monday(ts):
    ts = ts.normalize()
    return ts - pd.Timedelta(days=ts.weekday())

# -----------------------------
# 2) User-level cohort + start plan
# -----------------------------
user_start = (
    df.groupby("userid", as_index=False)
      .agg(start_effective=("startdate", "min"))
)

user_start["signup_week_start"] = user_start["start_effective"].apply(week_start_monday)
user_start["cohort_week"] = ((user_start["signup_week_start"] - BASE_WEEK_START) / pd.Timedelta(days=7)).astype(int)

user_start = user_start[user_start["cohort_week"].between(0, N_WEEKS-1)].copy()
df = df[df["userid"].isin(set(user_start["userid"]))].copy()

user_first_event = (
    df.sort_values(["userid", "date"])
      .groupby("userid", as_index=False)
      .first()[["userid", "plan"]]
      .rename(columns={"plan": "start_plan"})
)

user_first_event["start_plan"] = user_first_event["start_plan"].where(
    user_first_event["start_plan"].isin(["Free", "Paid"]), "Unknown"
)

user_dim = user_start.merge(user_first_event, on="userid", how="left")

# -----------------------------
# 3) Define Paid-starter W0 cohort
# -----------------------------
w0_paid_users = set(
    user_dim.loc[
        (user_dim["cohort_week"] == 0) & (user_dim["start_plan"] == "Paid"),
        "userid"
    ]
)

print("W0 Paid starters:", len(w0_paid_users))

# -----------------------------
# 4) Build end-of-week snapshots
# -----------------------------
week_ends = [BASE_WEEK_START + pd.Timedelta(days=6 + 7*i) for i in range(N_WEEKS)]
week_end_to_idx = {we: i for i, we in enumerate(week_ends)}

df["week_end"] = (
    df["date"]
    - pd.to_timedelta(df["date"].dt.weekday, unit="D")
    + pd.Timedelta(days=6)
).dt.normalize()

df = df[df["week_end"].isin(set(week_ends))].copy()
df["week_idx"] = df["week_end"].map(week_end_to_idx)

d = df[df["userid"].isin(w0_paid_users)].copy()
d = d.sort_values(["userid", "week_idx", "date"])

snap = (
    d.groupby(["userid", "week_idx"], as_index=False)
     .agg(
         end_week_plan=("plan", "last"),
         end_week_status=("Status", "last")
     )
)

# Expand full grid (userid x week)
grid = pd.MultiIndex.from_product(
    [sorted(w0_paid_users), range(N_WEEKS)],
    names=["userid", "week_idx"]
).to_frame(index=False)

snap = grid.merge(snap, on=["userid", "week_idx"], how="left")
snap = snap.sort_values(["userid", "week_idx"])

snap["end_week_plan"] = snap.groupby("userid")["end_week_plan"].ffill()
snap["end_week_status"] = snap.groupby("userid")["end_week_status"].ffill()

snap["end_week_plan"] = snap["end_week_plan"].where(
    snap["end_week_plan"].isin(["Free", "Paid"]), "Paid"
)
snap["end_week_status"] = snap["end_week_status"].where(
    snap["end_week_status"].isin(["Active", "Cancelled"]), "Active"
)

# -----------------------------
# 5) Final state classification
# -----------------------------
def classify_state(plan, status):
    if status == "Cancelled":
        return "Cancelled"
    if plan == "Free":
        return "Free"
    return "Paid"

def counts_for_week(w):
    s = snap[snap["week_idx"] == w].copy()
    s["state"] = [
        classify_state(p, st)
        for p, st in zip(s["end_week_plan"], s["end_week_status"])
    ]
    c = s["state"].value_counts()
    return (
        int(c.get("Paid", 0)),
        int(c.get("Free", 0)),
        int(c.get("Cancelled", 0))
    )

# -----------------------------
# 6) Sankey with dropdown
# -----------------------------
nodes = ["W0 Paid signups", "Paid", "Free", "Cancelled"]
node_idx = {n: i for i, n in enumerate(nodes)}

def sankey_links_for_week(w):
    paid_n, free_n, canc_n = counts_for_week(w)
    source = [node_idx["W0 Paid signups"]] * 3
    target = [node_idx["Paid"], node_idx["Free"], node_idx["Cancelled"]]
    value  = [paid_n, free_n, canc_n]
    return source, target, value

default_week = 5 if N_WEEKS > 5 else N_WEEKS - 1
src0, tgt0, val0 = sankey_links_for_week(default_week)

paid_n, free_n, canc_n = counts_for_week(default_week)
cohort_n = len(w0_paid_users)
cancel_rate = canc_n / cohort_n if cohort_n else np.nan

fig = go.Figure(data=[go.Sankey(
    node=dict(label=nodes, pad=18, thickness=18),
    link=dict(source=src0, target=tgt0, value=val0)
)])

buttons = []
for w in range(N_WEEKS):
    src, tgt, val = sankey_links_for_week(w)
    _, _, canc_n = counts_for_week(w)
    rate = canc_n / cohort_n if cohort_n else 0
    buttons.append(dict(
        label=f"End week W{w}",
        method="update",
        args=[
            {"link": [{"source": src, "target": tgt, "value": val}]},
            {"title": f"W0 Paid cohort → State at end of W{w} | Cancelled = {canc_n}/{cohort_n} ({rate*100:.1f}%)"}
        ]
    ))

fig.update_layout(
    title=f"W0 Paid cohort → State at end of W{default_week}",
    font_size=12,
    margin=dict(t=80, l=30, r=30, b=30),
    updatemenus=[dict(type="dropdown", x=0.0, y=1.1, buttons=buttons)]
)

fig.show()


W0 Paid starters: 8


In [40]:
'''
What is the relationship between feature usage and product retention or conversion to paid?
'''

'''
Step 1 — Setup + Define Outcomes

We define:

Retention outcome:

Retained = Active at end of Week 7

Churned = Cancelled at end of Week 7

Conversion outcome:

Started Free in W0

Converted if ever became Paid within first 4 weeks
'''

import pandas as pd
import numpy as np
from datetime import timedelta
import statsmodels.api as sm
import plotly.express as px
from scipy import stats


BASE_WEEK_START = pd.Timestamp("2025-03-03")
N_WEEKS = 8

df = pd.read_csv(DATA_PATH)

# -----------------------------
# 1. Clean
# -----------------------------
df["userid"] = df["userid"].astype(str).str.strip()
df["date"] = pd.to_datetime(df["date"])
df["startdate"] = pd.to_datetime(df["startdate"])

df["plan"] = df["plan"].str.title()
df["Status"] = df["Status"].str.title()

df = df.sort_values(["userid", "date"]).copy()

# -----------------------------
# 2. Build weekly end-of-week status
# -----------------------------
df["week_end"] = (
    df["date"] - pd.to_timedelta(df["date"].dt.weekday, unit="D") + pd.Timedelta(days=6)
).dt.normalize()

week_ends = [BASE_WEEK_START + timedelta(days=6 + 7*i) for i in range(N_WEEKS)]
week_end_to_idx = {we: i for i, we in enumerate(week_ends)}

df = df[df["week_end"].isin(set(week_ends))].copy()
df["week_idx"] = df["week_end"].map(week_end_to_idx)

snap = (
    df.groupby(["userid", "week_idx"], as_index=False)
      .agg(end_status=("Status","last"),
           end_plan=("plan","last"))
)

# expand full grid + forward fill
users = snap["userid"].unique()
grid = pd.MultiIndex.from_product([users, range(N_WEEKS)], names=["userid","week_idx"]).to_frame(index=False)

snap = grid.merge(snap, on=["userid","week_idx"], how="left")
snap = snap.sort_values(["userid","week_idx"])

snap["end_status"] = snap.groupby("userid")["end_status"].ffill()
snap["end_plan"] = snap.groupby("userid")["end_plan"].ffill()

# -----------------------------
# 3. Define Retention Outcome
# -----------------------------
retention = snap[snap["week_idx"] == 7][["userid","end_status"]].copy()
retention["retained"] = (retention["end_status"] == "Active").astype(int)

# -----------------------------
# 4. Compute Feature Usage (First 4 weeks)
# -----------------------------
df["DaysSinceStart"] = (df["date"] - df["startdate"]).dt.days

usage = (
    df[df["DaysSinceStart"].between(0, 27)]
      .groupby("userid")
      .agg(
          auto_usage=("AI_AutoComplete_Success","sum"),
          chat_usage=("AI_Chat_Success","sum"),
          agent_usage=("AI_Agent_Success","sum")
      )
      .reset_index()
)

data = retention.merge(usage, on="userid", how="left").fillna(0)


In [47]:
# Step 2 — General Comparison (Group A vs B)
summary = data.groupby("retained")[["auto_usage","chat_usage","agent_usage"]].mean()
summary


,auto_usage,chat_usage,agent_usage
retained,,,
0,105.285714,39.523810,19.476190
1,107.317726,42.327759,21.364548


In [48]:
'''
create side-by-side bar plot (NOT stacked) for comparison of average feature usage between retained and churned users from summary table, where group 0 is churned and group 1 is retained.
'''


import plotly.express as px

# Create readable group labels
data["group"] = data["retained"].map({
    1: "Retained (Active W7)",
    0: "Churned (Cancelled W7)"
})

# Compute average usage by group
summary = (
    data.groupby("group")[["auto_usage","chat_usage","agent_usage"]]
        .mean()
        .reset_index()
)

# Reshape for plotting
plot_df = summary.melt(
    id_vars="group",
    var_name="feature",
    value_name="avg_usage"
)

# Side-by-side grouped bar chart
fig = px.bar(
    plot_df,
    x="feature",
    y="avg_usage",
    color="group",
    barmode="group",   # <-- KEY CHANGE (side-by-side)
    title="Average Feature Usage: Retained vs Churned Users",
    labels={
        "feature": "Feature",
        "avg_usage": "Average Usage (First 4 Weeks)",
        "group": "User Group"
    }
)

fig.update_layout(
    xaxis_title="Feature",
    yaxis_title="Average Usage",
    legend_title="Group",
    margin=dict(t=60)
)

fig.show()




In [49]:
# Step 3 (Modified) — Weighted Logistic

import numpy as np
import statsmodels.api as sm
from sklearn.utils.class_weight import compute_class_weight

# Prepare X and y
X = data[["auto_usage", "chat_usage", "agent_usage"]]
X = sm.add_constant(X)
y = data["retained"]

# Compute class weights
classes = np.array([0,1])
weights = compute_class_weight(
    class_weight="balanced",
    classes=classes,
    y=y
)

class_weight_dict = dict(zip(classes, weights))

# Assign sample weights
sample_weights = y.map(class_weight_dict)

# Fit weighted logistic regression
model_weighted = sm.Logit(y, X).fit(weights=sample_weights)

print(model_weighted.summary())


Optimization terminated successfully.
         Current function value: 0.237582
         Iterations 7
                           Logit Regression Results                           
Dep. Variable:               retained   No. Observations:                  320
Model:                          Logit   Df Residuals:                      316
Method:                           MLE   Df Model:                            3
Date:                Tue, 24 Feb 2026   Pseudo R-squ.:                 0.01895
Time:                        10:21:08   Log-Likelihood:                -76.026
converged:                       True   LL-Null:                       -77.495
Covariance Type:            nonrobust   LLR p-value:                    0.4013
                  coef    std err          z      P>|z|      [0.025      0.975]
-------------------------------------------------------------------------------
const           2.4923      0.551      4.526      0.000       1.413       3.572
auto_usage     -0.0217    

/Users/cloudmaple/.pyenv/versions/3.11.2/lib/python3.11/site-packages/statsmodels/base/optimizer.py:18: FutureWarning:

Keyword arguments have been passed to the optimizer that have no effect. The list of allowed keyword arguments for method newton is: tol, ridge_factor. The list of unsupported keyword arguments passed include: weights. After release 0.14, this will raise.



In [50]:
np.exp(model_weighted.params)


const          12.088924
auto_usage      0.978515
chat_usage      1.031959
agent_usage     1.059838
dtype: float64

In [56]:
# What is the relationship between feature errors and product retention or conversion to paid?

# Step 1 — Prepare Error Features

import pandas as pd
import numpy as np
import statsmodels.api as sm
import plotly.express as px
from sklearn.utils.class_weight import compute_class_weight

BASE_WEEK_START = pd.Timestamp("2025-03-03")

df = pd.read_csv(DATA_PATH)

df["userid"] = df["userid"].astype(str).str.strip()
df["date"] = pd.to_datetime(df["date"])
df["startdate"] = pd.to_datetime(df["startdate"])
df["plan"] = df["plan"].str.title()
df["Status"] = df["Status"].str.title()

df = df.sort_values(["userid", "date"]).copy()

# Days since start
df["DaysSinceStart"] = (df["date"] - df["startdate"]).dt.days

# First 4 weeks usage window
error_usage = (
    df[df["DaysSinceStart"].between(0, 27)]
      .groupby("userid")
      .agg(
          auto_error=("AI_AutoComplete_Error","sum"),
          chat_error=("AI_Chat_Error","sum"),
          agent_error=("AI_Agent_Error","sum")
      )
      .reset_index()
)


In [58]:
# Step 2 — Define Retention Outcome (Week 7)
# End-of-week status
df["week_end"] = (
    df["date"] - pd.to_timedelta(df["date"].dt.weekday, unit="D") + pd.Timedelta(days=6)
).dt.normalize()

week7 = BASE_WEEK_START + pd.Timedelta(days=6 + 7*7)

retention = (
    df[df["week_end"] == week7]
      .sort_values(["userid","date"])
      .groupby("userid")
      .last()[["Status"]]
      .reset_index()
)

retention["retained"] = (retention["Status"] == "Active").astype(int)

data = retention.merge(error_usage, on="userid", how="left").fillna(0)


In [61]:
# Step 3 — Side-by-Side Visual (Errors vs Retention)
data["group"] = data["retained"].map({
    1: "Retained (Active W7)",
    0: "Churned (Cancelled W7)"
})

summary = (
    data.groupby("group")[["auto_error","chat_error","agent_error"]]
        .mean()
        .reset_index()
)

plot_df = summary.melt(id_vars="group",
                       var_name="feature",
                       value_name="avg_errors")

fig = px.bar(
    plot_df,
    x="feature",
    y="avg_errors",
    color="group",
    barmode="group",
    title="Average Feature Errors (First 4 Weeks): Retained vs Churned"
)

fig.show()


In [62]:
# Step 4 — Weighted Logistic Regression (Retention ~ Errors)
X = data[["auto_error","chat_error","agent_error"]]
X = sm.add_constant(X)
y = data["retained"]

# Class weights
classes = np.array([0,1])
weights = compute_class_weight("balanced", classes=classes, y=y)
weight_map = dict(zip(classes, weights))
sample_weights = y.map(weight_map)

model_error = sm.Logit(y, X).fit(weights=sample_weights)
print(model_error.summary())


Optimization terminated successfully.
         Current function value: 0.241603
         Iterations 7
                           Logit Regression Results                           
Dep. Variable:               retained   No. Observations:                  320
Model:                          Logit   Df Residuals:                      316
Method:                           MLE   Df Model:                            3
Date:                Tue, 24 Feb 2026   Pseudo R-squ.:                0.002349
Time:                        10:58:17   Log-Likelihood:                -77.313
converged:                       True   LL-Null:                       -77.495
Covariance Type:            nonrobust   LLR p-value:                    0.9476
                  coef    std err          z      P>|z|      [0.025      0.975]
-------------------------------------------------------------------------------
const           2.2127      1.208      1.832      0.067      -0.155       4.580
auto_error      0.0129    

/Users/cloudmaple/.pyenv/versions/3.11.2/lib/python3.11/site-packages/statsmodels/base/optimizer.py:18: FutureWarning:

Keyword arguments have been passed to the optimizer that have no effect. The list of allowed keyword arguments for method newton is: tol, ridge_factor. The list of unsupported keyword arguments passed include: weights. After release 0.14, this will raise.



In [63]:
# Step 5 — Conversion to Paid (Free Starters Only)
# Identify Free starters
first_plan = (
    df.groupby("userid").first()["plan"].reset_index()
)
free_users = first_plan[first_plan["plan"]=="Free"]["userid"]

converted = (
    df[(df["userid"].isin(free_users)) &
       (df["DaysSinceStart"].between(0,27))]
      .groupby("userid")["plan"]
      .apply(lambda x: (x=="Paid").any())
      .reset_index(name="converted")
)

conv_data = error_usage.merge(converted, on="userid", how="inner")

X2 = sm.add_constant(conv_data[["auto_error","chat_error","agent_error"]])
y2 = conv_data["converted"]

weights2 = compute_class_weight("balanced",
                                 classes=np.array([0,1]),
                                 y=y2)

weight_map2 = dict(zip([0,1], weights2))
sample_weights2 = y2.map(weight_map2)

model_conv_error = sm.Logit(y2, X2).fit(weights=sample_weights2)
print(model_conv_error.summary())


Optimization terminated successfully.
         Current function value: 0.602150
         Iterations 5
                           Logit Regression Results                           
Dep. Variable:              converted   No. Observations:                  225
Model:                          Logit   Df Residuals:                      221
Method:                           MLE   Df Model:                            3
Date:                Tue, 24 Feb 2026   Pseudo R-squ.:                0.004890
Time:                        10:59:15   Log-Likelihood:                -135.48
converged:                       True   LL-Null:                       -136.15
Covariance Type:            nonrobust   LLR p-value:                    0.7217
                  coef    std err          z      P>|z|      [0.025      0.975]
-------------------------------------------------------------------------------
const           1.0127      0.792      1.278      0.201      -0.540       2.565
auto_error      0.0009    

/Users/cloudmaple/.pyenv/versions/3.11.2/lib/python3.11/site-packages/statsmodels/base/optimizer.py:18: FutureWarning:

Keyword arguments have been passed to the optimizer that have no effect. The list of allowed keyword arguments for method newton is: tol, ridge_factor. The list of unsupported keyword arguments passed include: weights. After release 0.14, this will raise.



In [64]:
# When are users most likely to convert? What does their user behavioral user journey look like?
'''
0) Setup + cohort definitions

Free starter: user’s earliest observed plan == “Free”

Converted (within 4 weeks): user has any day with plan == “Paid” during days 0–27

Time-to-convert: first day index where plan becomes Paid (0–27). If never → NaN.
'''

import pandas as pd
import numpy as np
import plotly.express as px
import plotly.graph_objects as go
from scipy import stats
import statsmodels.api as sm

df = pd.read_csv(DATA_PATH)

# Clean
df["userid"] = df["userid"].astype(str).str.strip()
df["date"] = pd.to_datetime(df["date"], errors="coerce", format="mixed")
df["startdate"] = pd.to_datetime(df["startdate"], errors="coerce", format="mixed")
df["plan"] = df["plan"].astype(str).str.strip().str.title()
df["Status"] = df["Status"].astype(str).str.strip().str.title()

for c in ["AI_Chat_Success","AI_Chat_Error","AI_Agent_Success","AI_Agent_Error"]:
    df[c] = pd.to_numeric(df[c], errors="coerce")

df = df.dropna(subset=["userid","date","startdate"]).sort_values(["userid","date"]).copy()

# Day index
df["day"] = (df["date"] - df["startdate"]).dt.days

# Restrict analysis window to first 4 weeks
df4 = df[df["day"].between(0, 27)].copy()

# Identify Free starters by earliest observed plan
first_plan = (
    df.sort_values(["userid","date"])
      .groupby("userid", as_index=False)
      .first()[["userid","plan"]]
      .rename(columns={"plan":"start_plan"})
)
free_users = set(first_plan.loc[first_plan["start_plan"]=="Free","userid"])

df4 = df4[df4["userid"].isin(free_users)].copy()

# Conversion label + time-to-convert (first day when plan becomes Paid)
conv_day = (
    df4[df4["plan"]=="Paid"]
      .groupby("userid")["day"]
      .min()
      .rename("convert_day")
      .reset_index()
)

user_labels = pd.DataFrame({"userid": sorted(free_users)})
user_labels = user_labels.merge(conv_day, on="userid", how="left")
user_labels["converted_4w"] = user_labels["convert_day"].notna().astype(int)


In [65]:
user_labels

,userid,convert_day,converted_4w
0,U0002,3.0,1
1,U0003,12.0,1
2,U0004,13.0,1
3,U0006,7.0,1
4,U0007,19.0,1
...,...,...,...
220,U0316,26.0,1
221,U0317,18.0,1
222,U0318,6.0,1
223,U0319,7.0,1


In [66]:
# 1) When are users most likely to convert?
conv_dist = user_labels.dropna(subset=["convert_day"]).copy()
fig = px.histogram(
    conv_dist,
    x="convert_day",
    nbins=28,
    title="Time to convert (Free starters) — Distribution of first Paid day (0–27)",
    labels={"convert_day":"Day since signup (first Paid day)"}
)
fig.show()


In [67]:
# B) Cumulative conversion curve (by day)
# Cumulative conversion by day: P(converted by day d)
N = len(user_labels)
days = np.arange(0, 28)

cdf = []
for d in days:
    c = (user_labels["convert_day"].notna() & (user_labels["convert_day"] <= d)).sum()
    cdf.append(c / N if N else np.nan)

fig = go.Figure()
fig.add_trace(go.Scatter(x=days, y=cdf, mode="lines+markers", name="Cumulative conversion"))
fig.update_layout(
    title="Cumulative conversion over first 4 weeks (Free starters)",
    xaxis_title="Day since signup",
    yaxis_title="Conversion rate (by day)"
)
fig.show()


In [69]:
# 2) Behavioral journey: weekly trends (usage + errors) for Chat & Agent
df4["week"] = (df4["day"] // 7).astype(int)  # 0..3
df4 = df4[df4["week"].between(0,3)].copy()

per_user_week = (
    df4.groupby(["userid","week"], as_index=False)
       .agg(
           chat_success=("AI_Chat_Success","sum"),
           chat_error=("AI_Chat_Error","sum"),
           agent_success=("AI_Agent_Success","sum"),
           agent_error=("AI_Agent_Error","sum"),
           active_days=("day","nunique")
       )
)

journey = per_user_week.merge(user_labels[["userid","converted_4w","convert_day"]], on="userid", how="left")
journey["group"] = journey["converted_4w"].map({1:"Converted", 0:"Not converted"})


In [72]:
# A) Plot trends (mean per user-week) — Chat success/errors
weekly_means = (
    journey.groupby(["group","week"], as_index=False)
           .agg(
               chat_success=("chat_success","mean"),
               chat_error=("chat_error","mean"),
               agent_success=("agent_success","mean"),
               agent_error=("agent_error","mean"),
           )
)

fig = px.line(
    weekly_means,
    x="week",
    y="chat_success",
    color="group",
    markers=True,
    title="Chat usage journey (avg per user per week): Converted vs Not converted",
    labels={"week":"Week since signup (0–3)", "chat_success":"Avg Chat Success"}
)
fig.show()

fig = px.line(
    weekly_means,
    x="week",
    y="chat_error",
    color="group",
    markers=True,
    title="Chat errors journey (avg per user per week): Converted vs Not converted",
    labels={"week":"Week since signup (0–3)", "chat_error":"Avg Chat Errors"}
)
fig.show()


In [73]:
# B) Plot trends — Agent success/errors
fig = px.line(
    weekly_means,
    x="week",
    y="agent_success",
    color="group",
    markers=True,
    title="Agent usage journey (avg per user per week): Converted vs Not converted",
    labels={"week":"Week since signup (0–3)", "agent_success":"Avg Agent Success"}
)
fig.show()

fig = px.line(
    weekly_means,
    x="week",
    y="agent_error",
    color="group",
    markers=True,
    title="Agent errors journey (avg per user per week): Converted vs Not converted",
    labels={"week":"Week since signup (0–3)", "agent_error":"Avg Agent Errors"}
)
fig.show()


In [74]:
# C) Optional: error rates (more meaningful than raw errors)

journey["chat_error_rate"] = journey["chat_error"] / (journey["chat_success"] + journey["chat_error"] + 1e-9)
journey["agent_error_rate"] = journey["agent_error"] / (journey["agent_success"] + journey["agent_error"] + 1e-9)

rate_means = (
    journey.groupby(["group","week"], as_index=False)
           .agg(chat_error_rate=("chat_error_rate","mean"),
                agent_error_rate=("agent_error_rate","mean"))
)

fig = px.line(rate_means, x="week", y="agent_error_rate", color="group", markers=True,
              title="Agent error rate journey: Converted vs Not converted",
              labels={"week":"Week since signup (0–3)", "agent_error_rate":"Avg Agent error rate"})
fig.show()


In [76]:
# 3) Models to quantify relationship (what I would use)
'''
Model A: Discrete-time hazard model (best for “when do users convert?”)

This models the probability of conversion in week t given they haven’t converted before.

Outcome: convert_in_week (0/1)

Features: chat_success/errors, agent_success/errors (week t), and optionally lagged week t-1
'''
# Build a hazard dataset: each user contributes rows for weeks until they convert (or week 3)
haz = journey.copy()

# Convert week-based conversion time
haz["convert_week"] = (haz["convert_day"] // 7).astype("Int64")

# Event happens in the week equal to convert_week
haz["event"] = ((haz["convert_week"].notna()) & (haz["week"] == haz["convert_week"])).astype(int)

# Keep only weeks before or at conversion (risk set)
haz = haz[(haz["convert_week"].isna()) | (haz["week"] <= haz["convert_week"])].copy()

# Features (you can add lag features too)
X = haz[["week","chat_success","chat_error","agent_success","agent_error"]].copy()
X = sm.add_constant(X)
y = haz["event"]

# Class weights for imbalance (event=1 is rarer)
classes = np.array([0,1])
w = {c: (len(y) / (2*(y==c).sum())) for c in classes}
weights = y.map(w)

haz_model = sm.GLM(y, X, family=sm.families.Binomial(), freq_weights=weights).fit()
print(haz_model.summary())

# Odds ratios (interpret as multiplicative effect on hazard)
print(np.exp(haz_model.params))



                 Generalized Linear Model Regression Results                  
Dep. Variable:                  event   No. Observations:                  618
Model:                            GLM   Df Residuals:                      612
Model Family:                Binomial   Df Model:                            5
Link Function:                  Logit   Scale:                          1.0000
Method:                          IRLS   Log-Likelihood:                -424.31
Date:                Tue, 24 Feb 2026   Deviance:                       848.62
Time:                        15:05:27   Pearson chi2:                     618.
No. Iterations:                     4   Pseudo R-squ. (CS):            0.01304
Covariance Type:            nonrobust                                         
                    coef    std err          z      P>|z|      [0.025      0.975]
---------------------------------------------------------------------------------
const            -0.6867      0.417     -1.647

In [77]:
'''
Model B: Weighted logistic regression (convert within 4 weeks)

Simpler: “Converted by 4 weeks (yes/no)” using aggregated features.
'''  

# Aggregate user-level features over first 2 weeks / 4 weeks
user_feats = (
    journey.groupby("userid", as_index=False)
           .agg(
               chat_s_4w=("chat_success","sum"),
               chat_e_4w=("chat_error","sum"),
               agent_s_4w=("agent_success","sum"),
               agent_e_4w=("agent_error","sum"),
           )
    .merge(user_labels[["userid","converted_4w"]], on="userid", how="left")
)

X = sm.add_constant(user_feats[["chat_s_4w","chat_e_4w","agent_s_4w","agent_e_4w"]])
y = user_feats["converted_4w"]

# Balanced weights
w = {0: len(y)/(2*(y==0).sum()), 1: len(y)/(2*(y==1).sum())}
weights = y.map(w)

conv_model = sm.GLM(y, X, family=sm.families.Binomial(), freq_weights=weights).fit()
print(conv_model.summary())
print("Odds ratios:\n", np.exp(conv_model.params))


                 Generalized Linear Model Regression Results                  
Dep. Variable:           converted_4w   No. Observations:                  225
Model:                            GLM   Df Residuals:                      220
Model Family:                Binomial   Df Model:                            4
Link Function:                  Logit   Scale:                          1.0000
Method:                          IRLS   Log-Likelihood:                -154.45
Date:                Tue, 24 Feb 2026   Deviance:                       308.89
Time:                        15:11:18   Pearson chi2:                     225.
No. Iterations:                     4   Pseudo R-squ. (CS):            0.01336
Covariance Type:            nonrobust                                         
                 coef    std err          z      P>|z|      [0.025      0.975]
------------------------------------------------------------------------------
const          1.2339      1.409      0.876      0.3

In [81]:
'''
4) A/B tests to confirm findings (what I would run)

Since we don’t have a true randomized experiment, we can do pseudo A/B to validate directional results.

A/B Test 1: High Agent usage vs Low Agent usage (Week 0)

Group A: top 50% agent_success in week 0

Group B: bottom 50%

Compare conversion rates with a 2-proportion z-test
'''
from statsmodels.stats.proportion import proportions_ztest

w0 = journey[journey["week"] == 0].copy()
w0 = w0[["userid","agent_success","agent_error"]].merge(user_labels[["userid","converted_4w"]], on="userid", how="left")

threshold = w0["agent_success"].median()
w0["high_agent"] = (w0["agent_success"] > threshold).astype(int)

A = w0[w0["high_agent"]==1]["converted_4w"]
B = w0[w0["high_agent"]==0]["converted_4w"]

count = np.array([A.sum(), B.sum()])
nobs = np.array([len(A), len(B)])

z, p = proportions_ztest(count, nobs)
print("A/B test (High vs Low agent_success in W0):")
print("Conversion rate A:", A.mean(), "Conversion rate B:", B.mean(), "p-value:", p)


A/B test (High vs Low agent_success in W0):
Conversion rate A: 0.654320987654321 Conversion rate B: 0.7361111111111112 p-value: 0.1958598900138976


In [82]:
# A/B Test 2: Low error rate vs High error rate
w0["agent_error_rate"] = w0["agent_error"] / (w0["agent_success"] + w0["agent_error"] + 1e-9)
thr = w0["agent_error_rate"].median()
w0["low_error"] = (w0["agent_error_rate"] <= thr).astype(int)

A = w0[w0["low_error"]==1]["converted_4w"]  # low error
B = w0[w0["low_error"]==0]["converted_4w"]  # high error

count = np.array([A.sum(), B.sum()])
nobs = np.array([len(A), len(B)])
z, p = proportions_ztest(count, nobs)

print("A/B test (Low vs High agent_error_rate in W0):")
print("Conversion rate low-error:", A.mean(), "high-error:", B.mean(), "p-value:", p)


A/B test (Low vs High agent_error_rate in W0):
Conversion rate low-error: 0.7226890756302521 high-error: 0.6886792452830188 p-value: 0.5759510296882311
